# Issue #6: Segment all evaluation subsets with YAP

**Expected runtime:** 1–4 hours (QA contexts dominate). The segment cache makes
re-runs resume where they stopped — mount Drive (optional cell) to survive
runtime resets.

Prereq: the `issue-6-segment-subsets` branch (with `segmentation/segment_subsets.py`) is pushed.

In [ ]:
from getpass import getpass
token = getpass('GitHub PAT: ')
%cd /content
!rm -rf /content/NLP-Final-
!git clone https://{token}@github.com/AdonZahavi/NLP-Final-.git /content/NLP-Final-
%cd /content/NLP-Final-
!git checkout issue-6-segment-subsets
!ls data/subsets/

In [ ]:
# OPTIONAL but recommended: persist the segment cache on Drive so a runtime
# reset doesn't lose hours of YAP work.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/nlp_final_cache /content/NLP-Final-/cache
!rm -rf /content/NLP-Final-/cache
!ln -s /content/drive/MyDrive/nlp_final_cache /content/NLP-Final-/cache
!ls -la /content/NLP-Final-/cache/

## Build & start YAP (same recipe as issue #5)

In [ ]:
!apt-get update -qq && apt-get install -y -qq golang-go bzip2 > /dev/null 2>&1
import os
os.environ['GOPATH'] = '/content/gopath'
os.environ['GO111MODULE'] = 'off'

%cd /content
!rm -rf /content/gopath
!mkdir -p /content/gopath/src
!git clone -q https://github.com/OnlpLab/yap.git /content/gopath/src/yap
%cd /content/gopath/src/yap
!bunzip2 -k data/*.bz2
!git clone -q --depth 1 https://github.com/gorilla/mux.git vendor/github.com/gorilla/mux
!rm -rf vendor/github.com/gorilla/mux/.git
!mkdir -p vendor/gopkg.in
!git clone -q --depth 1 --branch v2 https://github.com/go-yaml/yaml.git vendor/gopkg.in/yaml.v2
!rm -rf vendor/gopkg.in/yaml.v2/.git
!ln -sf data/bgulex/bgupreflex_withdef.utf8.hr .
!ln -sf data/bgulex/bgulex.utf8.hr .
!go build -o /content/gopath/src/yap/yap_bin .
!ls -la /content/gopath/src/yap/yap_bin

In [ ]:
# Start YAP API server — file log + liveness polling
import subprocess, time, urllib.request, json

logf = open('/content/yap.log', 'w')
yap_proc = subprocess.Popen(
    ['./yap_bin', 'api'],
    cwd='/content/gopath/src/yap',
    stdout=logf, stderr=subprocess.STDOUT,
)
req = urllib.request.Request(
    'http://localhost:8000/yap/heb/joint',
    data=json.dumps({'text': 'שלום  '}).encode(),
    headers={'Content-Type': 'application/json'},
)
for attempt in range(60):
    if yap_proc.poll() is not None:
        print(f'YAP EXITED (code {yap_proc.returncode}). Log tail:')
        print(open('/content/yap.log').read()[-3000:])
        break
    try:
        with urllib.request.urlopen(req, timeout=120) as r:
            print(f'YAP ready (attempt {attempt+1})')
            break
    except Exception as e:
        print(f'  waiting... ({attempt+1}/60, {type(e).__name__})')
        time.sleep(10)
else:
    print('Timed out. Log tail:')
    print(open('/content/yap.log').read()[-3000:])

## Segment

In [ ]:
# Smoke test: 3 sentiment records
%cd /content/NLP-Final-
!python segmentation/segment_subsets.py --limit 3 --task sentiment
!head -c 1200 data/subsets/segmented/sentiment_500.jsonl

In [ ]:
# FULL RUN (1–4h). Resumable: if the runtime dies, re-run all cells — the
# Drive-mounted cache skips everything already segmented.
%cd /content/NLP-Final-
!python segmentation/segment_subsets.py

## Validate

In [ ]:
!pip install -q pandas
%cd /content/NLP-Final-
!python segmentation/gold_check.py

In [ ]:
# Peek at the manual QC sample
!head -60 data/subsets/segmented/qc_sample.md

## Commit results to the branch

In [ ]:
%cd /content/NLP-Final-
!git config user.email "orna.zahavi1@gmail.com" && git config user.name "Or Zahavi"
!git add data/subsets/segmented/
!git commit -m "Issue #6: YAP-segmented subsets + QC sample"
!git push origin issue-6-segment-subsets